# Exploratory customer-targeting checks

Naomi Feitong Liu | Customer targeting and revenue uncertainty.

This notebook is a readable guide to the later analyses and saved outputs. The original analysis remains in `01_targeting_analysis.ipynb`. All extensions here were developed after the original test outcomes were examined and remain exploratory.

**Default behavior:** review saved outputs and check their consistency. Set `RECOMPUTE=True` below, and supply the verified CSV, to rerun the listed analysis scripts. Opening this notebook does not constitute a new experiment or an independent confirmation.

**Colab setup:** the first cell downloads the public repository if its supporting files are absent. This requires the companion files to have been uploaded to GitHub. It does not download the raw customer CSV or refit models. In Colab, the revenue and audience review cells also display the figure files directly.

In [1]:
from pathlib import Path
import json
import subprocess
import sys
import numpy as np
import pandas as pd

# A GitHub-hosted Colab notebook does not automatically include sibling files.
candidates = [
    Path.cwd(), Path.cwd().parent, Path.cwd() / "writing_sample",
    Path.cwd() / "customer-targeting-reliability",
    Path("/content/customer-targeting-reliability"),
]
ROOT = next((p for p in candidates if (p / "analysis/results/original_notebook").is_dir()), None)
if ROOT is None:
    try:
        import google.colab
    except ImportError:
        raise FileNotFoundError(
            "Download the complete repository, then run this notebook from its notebooks folder."
        ) from None
    checkout = Path("/content/customer-targeting-reliability")
    if checkout.exists():
        raise FileNotFoundError(
            "The Colab checkout lacks the later analysis files. Restart the runtime after the GitHub upload."
        )
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/naomiliu119/customer-targeting-reliability.git",
        str(checkout),
    ], check=True)
    ROOT = checkout
    if not (ROOT / "analysis/results/original_notebook").is_dir():
        raise FileNotFoundError("Upload the companion analysis files to GitHub before running notebook 02.")

RESULTS = ROOT / "analysis/results"
EXPLORATORY = ROOT / "exploratory_checks_2026_10_28/results"
RECOMPUTE = False
DATA_PATH = ROOT / "data/hillstrom.csv"  # Change if your verified CSV is elsewhere.
print("Mode:", "recompute analyses" if RECOMPUTE else "review saved outputs")
print("Python:", sys.version.split()[0], "NumPy:", np.__version__, "pandas:", pd.__version__)


Mode: review saved outputs
Python: 3.12.14 NumPy: 2.3.5 pandas: 2.2.3


## 1. Optional regeneration

The raw data are not included. Obtain the CSV from the source documented in the original repository's `data/README.md`. The scripts verify the expected dataset where specified. Model settings, contact rates, and message rules are kept as documented; no search for more favorable test outcomes is performed.

The first script refits the two ridge models across 200 training resamples. The later scripts run the ridge-grid and message-matching checks, then the documented boundary and bootstrap follow-up. Their complete designs are recorded in the analysis plan. These commands require NumPy and pandas.

In [2]:
if RECOMPUTE:
    assert DATA_PATH.exists(), "Set DATA_PATH to the downloaded CSV."
    commands = [
        ["analysis/targeting_stability_check.py", str(DATA_PATH), "--output-dir", "analysis/results"],
        ["exploratory_checks_2026_10_28/run_checks.py", str(DATA_PATH)],
        ["exploratory_checks_2026_10_28/followup_checks.py", str(DATA_PATH)],
    ]
    for command in commands:
        subprocess.run([sys.executable, *command], cwd=ROOT, check=True)
else:
    print("Saved-output review only: no models were refitted by this cell.")

Saved-output review only: no models were refitted by this cell.


## 2. Original revenue comparison at equal contact rates

Figure 1 plots the advantage over **expected random targeting**, not total spending or profit. Each rule selects 2,130 of 8,523 test customers at the nominal 25% rate. The incremental-revenue row is the original primary comparison. The displayed approximate intervals hold fitted models and selections fixed.

In [3]:
original = pd.read_csv(RESULTS / "original_notebook/final_test_comparisons.csv")
main = original.loc[original["Contact rate"] == "25%"].copy()
assert len(main) == 4
assert ((main["95% CI lower"] < 0) & (main["95% CI upper"] > 0)).all()
print(main.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

if "google.colab" in sys.modules:
    from IPython.display import Image, display
    display(Image(filename=str(ROOT / "figures" / "revenue_comparisons.png")))


Contact rate                 Rule  Additional revenue  Advantage over random  95% CI lower  95% CI upper
         25%  Historical spending              379.19                 212.62        -70.40        495.65
         25% Purchase probability              351.05                 184.48       -141.25        510.21
         25%    Predicted revenue              350.89                 184.32       -139.71        508.35
         25%  Incremental revenue              341.84                 175.27        -72.45        422.99


![Revenue advantages over expected random targeting](../figures/revenue_comparisons.png)

Every interval crosses zero. This leaves the relative revenue benefit unresolved; it does not establish equivalence between rules.

## 3. Audience disagreement and sensitivity to training observations

Audience disagreement compares two different rules. Training sensitivity concerns the same modeling procedure fitted to resampled training observations. Neither measure alone establishes a revenue gain or loss.

In [4]:
checks = json.loads((RESULTS / "targeting_checks.json").read_text())
base = checks["independent_checks"]
stability = checks["new_exploratory_training_stability"]
frequencies = pd.read_csv(RESULTS / "selection_frequencies.csv")
original_selected = frequencies.loc[frequencies.original_selected, "selection_frequency"]
assert len(original_selected) == base["selected_per_rule"] == 2130
assert int((original_selected < 0.5).sum()) == 524
assert base["overlap_incremental_historical"] == 1288
print("Selected by each rule:", base["selected_per_rule"])
print("Shared selections:", base["overlap_incremental_historical"])
print("Unique to each rule:", base["replaced_per_rule"])
print("Originally selected, re-selected in fewer than half of refits:", int((original_selected < .5).sum()))
print("Retained audience share, 10th / median / 90th percentiles:",
      np.round(100*np.array(stability["retained_original_audience_fraction_quantiles_10_50_90"]), 1))

if "google.colab" in sys.modules:
    from IPython.display import Image, display
    display(Image(filename=str(ROOT / "figures" / "audience_and_training_sensitivity.png")))


Selected by each rule: 2130
Shared selections: 1288
Unique to each rule: 842
Originally selected, re-selected in fewer than half of refits: 524
Retained audience share, 10th / median / 90th percentiles: [54.8 70.8 82.5]


![Audience disagreement and training sensitivity](../figures/audience_and_training_sensitivity.png)

The unique groups in Panel A are different customers. Panel B includes only the 2,130 customers originally selected by incremental-revenue targeting; its horizontal axis is re-selection frequency, not purchase probability or model accuracy.

## 4. Ridge-setting sensitivity

Cross-validation uses training data only and selects penalties using spending prediction error. It does not optimize test revenue. The original alpha=1 model remains the baseline.

In [5]:
cv = pd.read_csv(EXPLORATORY / "ridge_training_cv.csv")
sensitivity = pd.read_csv(EXPLORATORY / "ridge_sensitivity.csv")
print("Training cross-validation:")
print(cv[["arm", "alpha", "out_of_fold_mse"]].to_string(index=False))
selected = sensitivity.loc[(sensitivity.partition == "test") & (sensitivity.contact_rate == .25) &
                           (sensitivity.setting.isin(["alpha_1", "training_CV_selected"]))]
print("\nOriginal and training-CV-selected rules, test evaluation:")
print(selected[["setting", "advantage_over_random", "advantage_ci_low", "advantage_ci_high", "retained_original_audience"]].to_string(index=False))

Training cross-validation:
        arm   alpha  out_of_fold_mse
  No E-Mail     0.1       156.798035
  No E-Mail     1.0       156.797981
  No E-Mail    10.0       156.797442
  No E-Mail   100.0       156.792541
  No E-Mail  1000.0       156.767699
  No E-Mail 10000.0       156.754562
Mens E-Mail     0.1       284.224777
Mens E-Mail     1.0       284.224527
Mens E-Mail    10.0       284.222062
Mens E-Mail   100.0       284.199782
Mens E-Mail  1000.0       284.097791
Mens E-Mail 10000.0       284.081089

Original and training-CV-selected rules, test evaluation:
             setting  advantage_over_random  advantage_ci_low  advantage_ci_high  retained_original_audience
             alpha_1             175.270430        -72.452214         422.993074                     1.00000
training_CV_selected             218.313959        -50.949590         487.577509                     0.81784


## 5. Matching message versions to merchandise history

These policies use all three randomized assignment groups and the **same fixed audience of 3,200 customers**. They evaluate version choice within that audience, not the optimal choice of whom to contact. Merchandise history does not identify gender. Customers with both category histories receive an expected 50/50 version mix under the matching rule.

In [6]:
values = pd.read_csv(EXPLORATORY / "message_policy_values.csv")
comparisons = pd.read_csv(EXPLORATORY / "message_policy_comparisons.csv")
nominal_values = values.loc[values.estimator == "nominal_IPW"]
nominal_comparisons = comparisons.loc[comparisons.estimator == "nominal_IPW"]
assert (nominal_values.selected == 3200).all()
assert ((nominal_comparisons.ci_low < 0) & (nominal_comparisons.ci_high > 0)).all()
print(nominal_values[["policy", "selected", "estimate"]].to_string(index=False))
print("\nPaired comparisons:")
print(nominal_comparisons[["comparison", "estimate", "ci_low", "ci_high", "bootstrap_low", "bootstrap_high"]].to_string(index=False))

                  policy  selected   estimate
         All men's email      3200 378.701664
       All women's email      3200 417.300211
 Expected 50/50 versions      3200 398.000937
History-matched versions      3200 230.408952

Paired comparisons:
                                   comparison    estimate      ci_low    ci_high  bootstrap_low  bootstrap_high
        History matched minus All men's email -148.292712 -394.292808  97.707385    -422.390028       75.477150
      History matched minus All women's email -186.891258 -589.646194 215.863677    -614.293649      181.606251
History matched minus Expected 50/50 versions -167.591985 -386.639380  51.455410    -399.473254       37.926793


## 6. Verification and figure generation

`analysis/audit_paper_figures.py` independently refits the original purchase and spending models and checks the plotted revenue points, intervals, and audience membership. It requires the original scientific-model dependencies, including scikit-learn and SciPy. The 200 training refits were also independently rerun during verification. This does not imply that the original full notebook or its percentile bootstrap was rerun in this notebook's default review mode.

In [7]:
audit_path = RESULTS / "figure_audit.json"
if audit_path.exists():
    audit = json.loads(audit_path.read_text())
    print("Saved independent audit:")
    print(pd.DataFrame(audit["figure_1_refitted_model_results"]).to_string(index=False))
    print("\nAudience checks:", audit["figure_2"])
else:
    print("No saved figure audit found. Run analysis/audit_paper_figures.py with the verified CSV.")
if RECOMPUTE:
    subprocess.run([sys.executable, "analysis/make_paper_figures.py"], cwd=ROOT, check=True)
else:
    print("\nFigures shown above are the saved analysis assets.")

Saved independent audit:
                rule  Additional revenue  Advantage over random  95% CI lower  95% CI upper
 Historical spending          379.190426             212.623475    -70.404222    495.651172
Purchase probability          351.045407             184.478456   -141.254215    510.211126
   Predicted revenue          350.888185             184.321234   -139.711002    508.353469
 Incremental revenue          341.837381             175.270430    -72.452214    422.993074

Audience checks: {'selected_per_rule': 2130, 'shared': 1288, 'unique_per_rule': 842, 'original_selected_below_half': 524, 'histogram_total': 2130}

Figures shown above are the saved analysis assets.


## Interpretation

The results document different contact allocations and training sensitivity without establishing a winning targeting rule. The matching comparison likewise does not establish an advantage for the specified matching rule. Future discount-versus-reminder experiments are proposed research, not results contained in this notebook.

For assumptions, estimators, and interpretation, read `ANALYSIS_README.md` in the repository root. For original model-development chronology, use notebook 01; the saved order of its cells is not independent preregistration.